### Structured Output

### Pydantic

In [ ]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-120b")
model


AIMessage(content='Here’s a concise overview of the most recent Marvel Cinematic Universe (MCU) film that was publicly released **as of my knowledge cutoff in\u202f2024**:\n\n| Item | Details |\n|------|---------|\n| **Title** | **Deadpool\u202f3** |\n| **Release date** | **May\u202f3\u202f2024** (United States) |\n| **Director** | **Shawn Levy** |\n| **Screenwriter(s)** | **Zack Stentz** (story) & **Jeremy Slater** (screenplay) |\n| **Production companies** | Marvel Studios, 20th\u202fCentury\u202fFox (now\u202f20th\u202fCentury\u202fStudios) |\n| **Running time** | Approximately **108\u202fminutes** |\n| **MPAA rating** | **R** (for strong violence, language, sexual content and drug use) |\n| **Key cast** | • **Ryan Reynolds** – Wade\u202fWilson / Deadpool  <br>• **Hugh Jackman** – Logan / Wolverine  <br>• **Jennifer Garner** – Elektra  <br>• **Emma Corrin** – Loki  <br>• **Jared Keeso** – Chris\u202fChamberlain (new character)  <br>• **James\u202fMarsden** – Captain\u202fAmerica (ca

In [2]:
from pydantic import BaseModel, Field

#Create schema
class Movie(BaseModel):
    title: str= Field (description="The title of the movie")
    year: int= Field (description="The year the movie was released")
    director: str= Field (description="The name of the directors of that movie")
    rating: float= Field (description="The movied ratings based on IDMB out of 10")
    cast: str= Field (description="The casts of the movie")

In [18]:
model_with_structure= model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000024CEA02E8B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000024CEA02F130>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'The name of the directors of that movie', 'type': 'string'}, 'rating': {'description': 'The m

In [16]:
model.invoke("Provide the details of the latest Marvel movie")

AIMessage(content='**Marvel’s most recent theatrical release (as of my knowledge cutoff\u202f–\u202fJune\u202f2024)**  \n\n| Item | Details |\n|------|---------|\n| **Title** | **Deadpool\u202f3** |\n| **Release date** | May\u202f3\u202f2024 (United States) |\n| **Running time** | ~108\u202fminutes |\n| **Director** | **Shawn Levy** (who also co‑produced *Free Guy* and *The Adam Project*) |\n| **Screenwriters** | **Paul Wernick** & **Rhett Reese** (returning from the first two Deadpool films) |\n| **Producers** | Kevin Feige (Marvel Studios), Ryan Reynolds, Shawn Levy, Lauren Shuler Donner, and others |\n| **Production companies** | Marvel Studios, 21\u202fLaps Entertainment, Maximum Effort |\n| **Distribution** | Walt Disney Studios Motion Pictures |\n| **Main cast** | • **Ryan Reynolds** as Wade Wilson / Deadpool  <br>• **Jennifer Garner** as **Eleven** (a new mutant‑type character) <br>• **Hugh Jackman** as **Logan / Wolverine** (his final live‑action appearance) <br>• **Emma Corrin

In [19]:
model_with_structure.invoke("Provide the details of the latest Marvel movie")

Movie(title='Guardians of the Galaxy Vol. 4', year=2026, director='James Gunn', rating=8.2, cast='Chris Pratt, Zoe Saldana, Dave Bautista, Karen Gillan, Pom Klementieff, Sean Gunn, Vin Diesel (voice of Groot), Bradley Cooper (voice of Rocket)')

### Message output alongside parsed structure

In [20]:
from pydantic import BaseModel, Field

#Create schema
class Movie(BaseModel):
    """A movie with details"""
    title: str= Field (..., description="The title of the movie")
    year: int= Field (..., description="The year the movie was released")
    director: str= Field (..., description="The name of the directors of that movie")
    rating: float= Field (..., description="The movied ratings based on IDMB out of 10")
    cast: str= Field (..., description="The casts of the movie")

model_with_structure= model.with_structured_output(Movie, include_raw=True)
response=model_with_structure.invoke("Provide the details of the latest Marvel movie")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants details of the latest Marvel movie. Need to know latest Marvel movie as of now (2026-04-25). Likely latest Marvel Cinematic Universe (MCU) release. As of 2026, upcoming movies: "Fantastic Four" maybe 2025, "Avengers: The Kang Dynasty" released 2025? Actually timeline: Avengers: Secret Wars 2026? Let\'s think.\n\nMarvel movies: "Guardians of the Galaxy Vol. 3" was 2023. Then "The Marvels" 2023. "Deadpool 3" 2024. "Captain America: New World Order" 2024. "Thunderbolts" 2024. "Blade" 2025. "Fantastic Four" 2025. "Avengers: The Kang Dynasty" 2025. "Avengers: Secret Wars" 2026. So latest released as of April 2026 could be "Avengers: Secret Wars" (released May 2026 maybe). But if not yet released, maybe "Fantastic Four" early 2025. Let\'s assume the latest released is "Avengers: Secret Wars". Provide details: title, year, director, cast, rating. Use function Movie. Need to call function with details. Provide ra

### Nested structure

In [23]:
from pydantic import BaseModel, Field
class Actor(BaseModel):
    name: str 
    role: str

class MovieDetails (BaseModel):
    title: str
    year: int
    cast:list[Actor]
    genres:list[str]
    budget: float | None = Field(None, description ='Budget in million taka')

In [24]:
model_with_structure= model.with_structured_output(MovieDetails)
response=model_with_structure.invoke("Provide the details of the latest Marvel movie")
response

MovieDetails(title='Avengers: Secret Wars', year=2025, cast=[Actor(name='Robert Downey Jr.', role='Tony Stark / Iron Man'), Actor(name='Chris Evans', role='Steve Rogers / Captain America'), Actor(name='Scarlett Johansson', role='Natasha Romanoff / Black Widow'), Actor(name='Chris Hemsworth', role='Thor'), Actor(name='Tom Holland', role='Peter Parker / Spider-Man'), Actor(name='Benedict Cumberbatch', role='Doctor Strange'), Actor(name='Brie Larson', role='Carol Danvers / Captain Marvel'), Actor(name='Paul Rudd', role='Scott Lang / Ant-Man'), Actor(name='Karen Gillan', role='Nebula'), Actor(name='Mahershala Ali', role='Mysterio / Quentin Beck'), Actor(name='Zendaya', role='MJ'), Actor(name='Samuel L. Jackson', role='Nick Fury')], genres=['Action', 'Adventure', 'Science Fiction'], budget=250000000.0)

### TypeDict

In [25]:
from typing_extensions import TypedDict, Annotated



In [26]:
#Create schema
class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [27]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions Taka")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'},
  {'name': 'Clark Gregg', 'role': 'Phil Coulson'},
  {'name': 'Cobie Smulders', 'role': 'Maria Hill'},
  {'name': 'Stellan Skarsgård', 'role': 'Dr. Erik Selvig'}],
 'genres': ['Action', 'Adventure', 'Science Fiction', 'Superhero'],
 'title': 'The Avengers',
 'year': 2012}

### Dataclasses

In [30]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='d62aafb3-d17a-422b-8552-47f8a96f180d'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'reasoning_content': 'The user wants extraction. According to system, we must output only final JSON object matching ContactInfo schema. Include name, email, phone. So:\n\n{\n"name":"John Doe",\n"email":"john@example.com",\n"phone":"(555) 123-4567"\n}\n\nNeed compact JSON.'}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 238, 'total_tokens': 331, 'completion_time': 0.191950844, 'completion_tokens_details': {'reasoning_tokens': 61}, 'prompt_time': 0.009724395, 'prompt_tokens_details': None, 'queue_time': 0.182740752, 'total_time': 0.201675239}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_3af2d41834', 'service_tier': 'on_demand', 'fin

In [31]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [34]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [35]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')